In [ ]:
# importing the required packages
import czifile # to import a .czi file
# from PIL import Image # to convert .czi file to a .tif file
import skimage # general package for manipulating imaging data
from pathlib import Path # for file path 
import numpy as np
import matplotlib.pyplot as plt
from microfilm.microplot import microshow # for viewing multichannel image data
from skimage.transform import rotate # to rotate the image as a control
from skimage.restoration import rolling_ball # for image processing
from skimage.filters import gaussian # for image processing
from skimage.feature import peak_local_max # for local max detection
import sys
import os

import optuna
import pandas as pd
import re

# with this peice of code, it will recognize the custom modules
project_root = "/Users/cgeyskens/Documents/code/phd/image-analysis/synapse-counting"
sys.path.append(project_root)

# custom modules
from synapse_counting import metadata, preprocessing, calc_synaptic_metrics

In [ ]:
def local_peak_detection(presynapse_preprocessed, postsynapse_preprocessed, presynapse_distance, postsynapse_distance, presynapse_threshold, postsynapse_threshold):
    
    """Detects the local intensity peak of each channel processed
    
        Args:
            vglut1_preprocessed (np.array): processed image of vlgut1, which is background substracted and has a gaussian blur
            psd95_preprocessed (np.array): processed image of psd95, which is background substracted and has a gaussian blur
            vglut1_distance (float): minimun distance between vglut1 local peak maxima, usually 1
            psd95_distance (float): mimimun distance between psd95 local peak maxima, usually 1
            vglut1_threshold (float): thresholding of the vlgut1 image for local peak detection
            psd95_threshold (float): thresholding of the psd95 image for local peak detection
    
        Returns:
            vglut1_coord (np.array): coordinates of vglut1 local peak maxima
            psd95_coord (np.array): coordinates of psd95 local peak maxima
            psd95_rot_coord (np.array): coordinates of psd95 local peak maxima
            
            plot of vglut1, psd95, psd95_rot images with the local peak maxima overlaid
    """
    
    # Thresholding the image for local peak maximum detection
    presynapse_coord = peak_local_max(presynapse_preprocessed, min_distance = presynapse_distance, threshold_abs = presynapse_threshold)
    presynapse_coord = peak_local_max(postsynapse_preprocessed, min_distance = postsynapse_distance, threshold_abs = postsynapse_threshold)

    # Rotating an image (psd95) as a control
    postsynapse_rot = rotate(postsynapse_preprocessed, 90)
    postsynapse_rot_coord = peak_local_max(postsynapse_rot, min_distance = postsynapse_distance, threshold_abs = postsynapse_threshold)
    
    # Showing the local peaks with coordinates together with the images
    # fig, axs = plt.subplots(1, 3, figsize=(30, 30))

    # axs[0].imshow(vglut1_preprocessed, cmap='gray')
    # axs[0].plot(vglut1_coord[:, 1], vglut1_coord[:, 0], 'c.')
    # axs[0].set_title('vglut1_pre')

    # axs[1].imshow(psd95_preprocessed, cmap='gray')
    # axs[1].plot(psd95_coord[:, 1], psd95_coord[:, 0], 'm.')
    # axs[1].set_title('psd95_pre')

    # axs[2].imshow(psd95_rot, cmap='gray')
    # axs[2].plot(psd95_rot_coord[:, 1], psd95_rot_coord[:, 0], 'm.')
    # axs[2].set_title('psd95_rot')
    
    return presynapse_coord, presynapse_coord, postsynapse_rot_coord

In [ ]:
from scipy.spatial.distance import cdist

def count_coloc_spots(presynapse_coordinates, postsynapse_coordinates, pixel_size_um, max_distance_um):
    
    """Counts the number of colocalized pre and postsynaptic spots based on the coordinates array of the local maxima detection
    
    Args:
        vlgut1_coordinates (np.array): coordinates of the local peak maxima in the vlgut1 channel
        psd95_coordinates (np.array): coordinates of the local peak maxima in the psd95 channel
        pixel_size_um (float): size of a pixel in um, based on image settings
        max_distance_um (float): value of the maximum colocalization distance between the spots of each channel in um
    
    Returns:
        colocalized spot count
    """
    
    # calculate the max distance in pixels with the pixel_size_um and max_distance_um
    max_distance_px = max_distance_um/pixel_size_um
    
    # Calculate pairwise distances between spots in vlgut1 and psd95 channel
    distances_pre_to_post = cdist(presynapse_coordinates, postsynapse_coordinates)
    distances_post_to_pre = cdist(postsynapse_coordinates, presynapse_coordinates)

    # Find unique colocalized spots
    colocalized_spots = set()

    # Iterate over distances from presynapse_coordinates to postsynapse_coordinates
    for i in range(len(presynapse_coordinates)):
        # Check if the current spot in vlgut1 has nearby spots in psd95
        colocalized_indices = [j for j, distance in enumerate(distances_pre_to_post[i, :]) if distance <= max_distance_px]
        for j in colocalized_indices:
            colocalized_spots.add((i, j))

    # Iterate over distances from postsynapse_coordinates to presynapse_coordinates
    for i in range(len(postsynapse_coordinates)):
        # Check if the current spot in Channel 2 has nearby spots in Channel 1
        colocalized_indices = [j for j, distance in enumerate(distances_post_to_pre[i, :]) if distance <= max_distance_px]
        for j in colocalized_indices:
            colocalized_spots.add((j, i))

    # Get the count of unique colocalized spots
    colocalized_spot_count = len(colocalized_spots)
    
    return colocalized_spot_count

In [ ]:
# input folder
input_folder = "/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images"

# get a list of files in that input_folder
file_list = os.listdir(input_folder)
print(file_list)

protein_and_synaptic_marker = "VCAM1_LacZ_VGLUT1_PSD95"

In [ ]:
import dask
from dask import delayed, compute
import dask.multiprocessing


In [ ]:
def load_and_preprocess(file_path, presynapse_channel, postsynapse_channel, preprocess_params):
    # extract metadata
    pixel_size_um, _ , _ = metadata.extract_metadata(file_path)
    # extract channels
    pre, post = preprocessing.extract_and_split(file_path, presynapse_channel = presynapse_channel, postsynapse_channel = postsynapse_channel)
    # preprocessing
    p = preprocessing.ImagePreprocessing(
        include_rolling_ball=preprocess_params['include_rolling_ball'], radius=preprocess_params['radius'],
        include_blur=preprocess_params['include_blur'], sigma=preprocess_params['sigma'], preserve_range=True,
        include_clahe=preprocess_params['include_clahe'],
        include_tophat=preprocess_params['include_tophat'], element_size=preprocess_params['element_size']
    )
    pre_processed, post_processed = p.preprocess(pre, post)
    return pre_processed, post_processed, pixel_size_um

In [ ]:
def objective(trial, presynapse_preprocessed, postsynapse_preprocessed, pixel_size_um, param_ranges):
    
    pre_distance = trial.suggest_int("pre_distance", *param_ranges['pre_distance'])
    post_distance = trial.suggest_int("post_distance", *param_ranges['post_distance'])
    pre_threshold = trial.suggest_float("pre_threshold", *param_ranges['pre_threshold'])
    post_threshold = trial.suggest_float("post_threshold", *param_ranges['post_threshold'])
    max_distance_um = trial.suggest_float("max_distance_um", *param_ranges['max_distance_um'])

    # probing the first function to get the parameters as input for the second function
    pre_coord, post_coord, post_rot_coord = local_peak_detection(presynapse_preprocessed = presynapse_preprocessed,
                                                                 postsynapse_preprocessed = postsynapse_preprocessed, 
                                                                 presynapse_distance = pre_distance, 
                                                                 postsynapse_distance = post_distance, 
                                                                 presynapse_threshold = pre_threshold, 
                                                                 postsynapse_threshold = post_threshold)
                                                                 
    # probing the second function where psd95 is not rotated, the actual condition
    colocalized_spot_count = count_coloc_spots(pre_coord, post_coord, pixel_size_um, max_distance_um)
    # probing the second function where psd95 is rotated, the internal control condition
    colocalized_spot_count_rot = count_coloc_spots(pre_coord, post_rot_coord, pixel_size_um, max_distance_um)
    
    # getting the scale differences between the spot count of the normal situation and psd95 rotated
    scaled_difference = (colocalized_spot_count - colocalized_spot_count_rot) / max(colocalized_spot_count, 1)
    
    return scaled_difference

In [ ]:
# main optimization process for each hippocampal_layer, without dask
def optimize_parameters_for_hippocampal_layer(file_list, input_folder, hippocampal_layers, preprocess_params_by_hippocampal_layer, param_ranges, nr_of_trials):
    
    best_params_by_hippocampal_layer = {}
    results_dfs = []
    all_trials_data = []
    
    for hippocampal_layer in hippocampal_layers:
        images = [f for f in file_list if hippocampal_layer in f and "LacZ-gRNA" in f] # only taking control (LacZ-images) images for setting the parameters
    
        preprocess_params = preprocess_params_by_hippocampal_layer[hippocampal_layer]
        
        # extract metadata and preprocess images
        image_results = []
        for file_name in images:
            file_path = os.path.join(input_folder, file_name)
            pre_1, post_1, pixel_size_um = load_and_preprocess(file_path, presynapse_channel = 0, postsynapse_channel = 1, preprocess_params = preprocess_params)
            if pre_1 is not None and post_1 is not None:
                image_results.append((pre_1, post_1, pixel_size_um, file_path))
        
        # optuna optimization
        def hippocampal_layer_objective(trial):
            detailed_trial_results = []
            for pre_1, post_1, pixel_size_um, file_path in image_results:
                score = objective(trial, pre_1, post_1, pixel_size_um, param_ranges[hippocampal_layer])
                detailed_trial_results.append({
                    'hippocampal_layer': hippocampal_layer,
                    'trial': trial.number,
                    'image': file_path,
                    'params': trial.params,
                    'score': score
                })
            mean_score = np.mean([result['score'] for result in detailed_trial_results])
            return mean_score, detailed_trial_results
        
        # create the study
        study = optuna.create_study(study_name=hippocampal_layer, direction="maximize", sampler=optuna.samplers.TPESampler())
        detailed_trials_data = []

        # define the optuna objective, such that the mean score of all the images in a certain hippocampal layer is optimized
        def optuna_objective(trial):
            mean_score, detailed_trial_results = hippocampal_layer_objective(trial)
            detailed_trials_data.extend(detailed_trial_results)
            return mean_score
        
        # do the optimization
        study.optimize(optuna_objective, n_trials = nr_of_trials)
    
        # store best parameters for the hippocampal_layer with the corresponding score
        best_trial_full = study.best_trial
        best_params_by_hippocampal_layer[hippocampal_layer] = {
            "best_trial": best_trial_full.number,
            "best_params": study.best_params,
            "best_score": study.best_value
            }
        
        # create dataframe for detailed trial results and the best params
        trials_df = pd.DataFrame(detailed_trials_data)
        results_dfs.append(trials_df)

        # getting the data per trial
        all_trials_data.extend([{'hippocampal_layer': hippocampal_layer} | trial for trial in study.trials_dataframe().to_dict('records')])

    # create dataframe for best parameters by hippocampal_layer
    best_params_by_hippocampal_layer_df = pd.DataFrame.from_dict(best_params_by_hippocampal_layer, orient = "index").reset_index()
    best_params_by_hippocampal_layer_df = pd.concat([best_params_by_hippocampal_layer_df.drop(['best_params'], axis=1), 
                                pd.json_normalize(best_params_by_hippocampal_layer_df['best_params'])], axis=1) # from json to columns

    # combine dataframes for all hippocampal_layers
    final_df = pd.concat(results_dfs, ignore_index=True)

    # combine the trial data
    final_df_optuna = pd.DataFrame(all_trials_data)

    return best_params_by_hippocampal_layer_df, final_df, final_df_optuna


In [ ]:
import os
import dask
from dask import delayed, compute
import numpy as np
import pandas as pd
import optuna

# main optimization process for each hippocampal_layer, with dask
def optimize_parameters_for_hippocampal_layer(file_list, input_folder, hippocampal_layers, preprocess_params_by_hippocampal_layer, param_ranges, nr_of_trials):
    
    @dask.delayed
    def process_hippocampal_layer(hippocampal_layer):
        images = [f for f in file_list if hippocampal_layer in f and "LacZ-gRNA" in f] # only taking control (LacZ-images) images for setting the parameters
    
        preprocess_params = preprocess_params_by_hippocampal_layer[hippocampal_layer]
        
        # extract metadata and preprocess images
        image_results = []
        for file_name in images:
            file_path = os.path.join(input_folder, file_name)
            pre_1, post_1, pixel_size_um = load_and_preprocess(file_path, presynapse_channel=0, postsynapse_channel=1, preprocess_params=preprocess_params)
            if pre_1 is not None and post_1 is not None:
                image_results.append((pre_1, post_1, pixel_size_um, file_path))
        
        # optuna optimization
        def hippocampal_layer_objective(trial):
            detailed_trial_results = []
            for pre_1, post_1, pixel_size_um, file_path in image_results:
                score = objective(trial, pre_1, post_1, pixel_size_um, param_ranges[hippocampal_layer])
                detailed_trial_results.append({
                    'hippocampal_layer': hippocampal_layer,
                    'trial': trial.number,
                    'image': file_path,
                    'params': trial.params,
                    'score': score
                })
            mean_score = np.mean([result['score'] for result in detailed_trial_results])
            return mean_score, detailed_trial_results
        
        # create the study
        study = optuna.create_study(study_name=hippocampal_layer, direction="maximize", sampler=optuna.samplers.TPESampler())
        detailed_trials_data = []

        # define the optuna objective, such that the mean score of all the images in a certain hippocampal layer is optimized
        def optuna_objective(trial):
            mean_score, detailed_trial_results = hippocampal_layer_objective(trial)
            detailed_trials_data.extend(detailed_trial_results)
            return mean_score
        
        # do the optimization
        study.optimize(optuna_objective, n_trials=nr_of_trials)
    
        # store best parameters for the hippocampal_layer with the corresponding score
        best_trial_full = study.best_trial
        best_params = {
            "best_trial": best_trial_full.number,
            "best_params": study.best_params,
            "best_score": study.best_value
        }
        
        # create dataframe for detailed trial results and the best params
        trials_df = pd.DataFrame(detailed_trials_data)
        
        # getting the data per trial
        trials_data = [{'hippocampal_layer': hippocampal_layer} | trial for trial in study.trials_dataframe().to_dict('records')]

        return hippocampal_layer, best_params, trials_df, trials_data

    # Collect the delayed tasks
    delayed_tasks = [process_hippocampal_layer(hippocampal_layer) for hippocampal_layer in hippocampal_layers]

    # Compute the results in parallel
    results = compute(*delayed_tasks)

    best_params_by_hippocampal_layer = {}
    results_dfs = []
    all_trials_data = []

    for hippocampal_layer, best_params, trials_df, trials_data in results:
        best_params_by_hippocampal_layer[hippocampal_layer] = best_params
        results_dfs.append(trials_df)
        all_trials_data.extend(trials_data)

    # create dataframe for best parameters by hippocampal_layer
    best_params_by_hippocampal_layer_df = pd.DataFrame.from_dict(best_params_by_hippocampal_layer, orient="index").reset_index()
    best_params_by_hippocampal_layer_df = pd.concat([best_params_by_hippocampal_layer_df.drop(['best_params'], axis=1), 
                                          pd.json_normalize(best_params_by_hippocampal_layer_df['best_params'])], axis=1) # from json to columns

    # combine dataframes for all hippocampal_layers
    final_df = pd.concat(results_dfs, ignore_index=True)

    # combine the trial data
    final_df_optuna = pd.DataFrame(all_trials_data)

    return best_params_by_hippocampal_layer_df, final_df, final_df_optuna


In [ ]:
# Example usage
input_folder = "/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images"
file_list = os.listdir(input_folder)

hippocampal_layers = ["CA1_SO", "CA1_SR", "CA1_SLM", "CA3_SO", "CA3_SL", "CA3_SR", "DG_Hilus", "DG_ML"]  # Example hippocampal_layers

preprocess_params_by_hippocampal_layer = {
    'CA1_SO': {
        'include_rolling_ball': True, 'radius': 5,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 10
    },
    'CA1_SR': {
        'include_rolling_ball': True, 'radius': 10,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 10
    },
    'CA1_SLM': {
        'include_rolling_ball': True, 'radius': 5,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 5
    },
    'CA3_SO': {
        'include_rolling_ball': True, 'radius': 5,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 10
    },
    'CA3_SL': {
        'include_rolling_ball': True, 'radius': 15,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 20
    },
    'CA3_SR': {
        'include_rolling_ball': True, 'radius': 10,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 10
    },
    'DG_Hilus': {
        'include_rolling_ball': True, 'radius': 15,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 20
    },
    'DG_ML': {
        'include_rolling_ball': True, 'radius': 5,
        'include_blur': True, 'sigma': 2,
        'include_clahe': False,
        'include_tophat': True, 'element_size': 15
    },
}

param_ranges = {
    'CA1_SO': {
        'pre_distance': (1, 5),
        'post_distance': (1,5),
        'pre_threshold': (500, 900),
        'post_threshold': (100, 150),
        'max_distance_um': (0.01, 1)
    },
    'CA1_SR': {
        'pre_distance': (1, 5),
        'post_distance': (1,5),
        'pre_threshold': (300, 500),
        'post_threshold': (100, 200),
        'max_distance_um': (0.01, 1)
    },
    'CA1_SLM': {
        'pre_distance': (1, 5),
        'post_distance': (1,5),
        'pre_threshold': (150, 250),
        'post_threshold': (50, 80),
        'max_distance_um': (0.01, 1)
    },
    'CA3_SO': {
        'pre_distance': (1, 5),
        'post_distance': (1,5),
        'pre_threshold': (500, 1200),
        'post_threshold': (100, 160),
        'max_distance_um': (0.01, 1)
    },
    'CA3_SL': {
        'pre_distance': (1, 10),
        'post_distance': (1, 10),
        'pre_threshold': (1000, 2000),
        'post_threshold': (250, 400),
        'max_distance_um': (0.01, 1)
    },
    'CA3_SR': {
        'pre_distance': (1, 5),
        'post_distance': (1, 5),
        'pre_threshold': (500, 1000),
        'post_threshold': (100, 200),
        'max_distance_um': (0.01, 1)
    },
    'DG_Hilus': {
        'pre_distance': (1, 10),
        'post_distance': (1, 10),
        'pre_threshold': (1000, 3000),
        'post_threshold': (300, 500),
        'max_distance_um': (0.01, 1)
    },
    'DG_ML': {
        'pre_distance': (1, 10),
        'post_distance': (1, 10),
        'pre_threshold': (500, 1000),
        'post_threshold': (100, 200),
        'max_distance_um': (0.01, 1)
    }
}

nr_of_trials = 10

best_params_by_hippocampal_layer_df, final_df, final_df_optuna = optimize_parameters_for_hippocampal_layer(file_list, input_folder, hippocampal_layers, preprocess_params_by_hippocampal_layer, param_ranges, nr_of_trials)

In [ ]:
best_params_by_hippocampal_layer_df

In [ ]:
file = "/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images/CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_DG_ML.czi"
pixel_size_um, _ , image_size_um = metadata.extract_metadata(file)
pre, post = preprocessing.extract_and_split(file, presynapse_channel = 0, postsynapse_channel = 1)

p = preprocessing.ImagePreprocessing(
            include_rolling_ball = True, radius = 5, # rolling ball parameters
            include_clahe = False, clip_limit = 0.005, kernel_size = 150, nbins = 265, # CLAHE parameters
            include_tophat = True, element_size = 15, # tophat parameters
            include_blur = True, sigma = 2, preserve_range = True # gaussian blur filters
            )
pre_1, post_1 = p.preprocess(pre, post)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize = (30, 30))
microshow(pre_1, ax=axs[0], label_text = 'VLGUT1')
microshow(post_1, ax=axs[1], label_text = 'PSD95')

In [ ]:
def local_peak_detection(vglut1_preprocessed, psd95_preprocessed, vglut1_distance, psd95_distance, vglut1_threshold, psd95_threshold):
    
    """Detects the local intensity peak of each channel processed
    
        Args:
            vglut1_preprocessed (np.array): processed image of vlgut1, which is background substracted and has a gaussian blur
            psd95_preprocessed (np.array): processed image of psd95, which is background substracted and has a gaussian blur
            vglut1_distance (float): minimun distance between vglut1 local peak maxima, usually 1
            psd95_distance (float): mimimun distance between psd95 local peak maxima, usually 1
            vglut1_threshold (float): thresholding of the vlgut1 image for local peak detection
            psd95_threshold (float): thresholding of the psd95 image for local peak detection
    
        Returns:
            vglut1_coord (np.array): coordinates of vglut1 local peak maxima
            psd95_coord (np.array): coordinates of psd95 local peak maxima
            psd95_rot_coord (np.array): coordinates of psd95 local peak maxima
            
            plot of vglut1, psd95, psd95_rot images with the local peak maxima overlaid
    """
    
    # Thresholding the image for local peak maximum detection
    vglut1_coord = peak_local_max(vglut1_preprocessed, min_distance = vglut1_distance, threshold_abs=vglut1_threshold)
    psd95_coord = peak_local_max(psd95_preprocessed, min_distance = psd95_distance, threshold_abs=psd95_threshold)

    # Rotating an image (psd95) as a control
    psd95_rot = rotate(psd95_preprocessed, 90)
    psd95_rot_coord = peak_local_max(psd95_rot, min_distance=psd95_distance, threshold_abs = psd95_threshold)
    
    # Showing the local peaks with coordinates together with the images
    fig, axs = plt.subplots(2, 2, figsize=(30, 30))

    axs[0,0].imshow(vglut1_preprocessed, cmap='gray')
    # axs[0,0].plot(vglut1_coord[:, 1], vglut1_coord[:, 0], 'c.')
    axs[0,0].set_title('vglut1_pre')

    axs[0,1].imshow(psd95_preprocessed, cmap='gray')
    # axs[0,1].plot(psd95_coord[:, 1], psd95_coord[:, 0], 'm.')
    axs[0,1].set_title('psd95_pre')

    axs[1,0].imshow(vglut1_preprocessed, cmap='gray')
    axs[1,0].plot(vglut1_coord[:, 1], vglut1_coord[:, 0], 'c.')
    axs[1,0].set_title('vglut1_pre')

    axs[1,1].imshow(psd95_preprocessed, cmap='gray')
    axs[1,1].plot(psd95_coord[:, 1], psd95_coord[:, 0], 'm.')
    axs[1,1].set_title('psd95_pre')


    # axs[2].imshow(psd95_rot, cmap='gray')
    # axs[2].plot(psd95_rot_coord[:, 1], psd95_rot_coord[:, 0], 'm.')
    # axs[2].set_title('psd95_rot')
    
    return vglut1_coord, psd95_coord, psd95_rot_coord

In [ ]:
# {'pre_distance': 2, 'post_distance': 3, 'pre_threshold': 134.82934781597908, 'post_threshold': 149.97743055799035, 'max_distance_um': 0.025655884409119978}

vglut1_coord, psd95_coord, psd95_rot_coord = local_peak_detection(pre_1, post_1, 7, 10, 756.556879, 192.641835)
print(vglut1_coord.shape, psd95_coord.shape, psd95_rot_coord.shape)

In [ ]:

# This block of code calculates the unique colocalized VGLUT1-PSD95 count, meaning it only has unique pairs of colocalized spots.
# It checks of every VLGUT1 and PSD95 spot, whether there are any PSD95 (for VGLUT1) and VLGUT1 (for PSD95) spots in their vicinity within a specific maximum distance and counts it. 
# So it can have for one spot, multiple pairs, and thus including multi-synaptic boutons.

from scipy.spatial.distance import cdist

def count_coloc_spots(vglut1_coordinates, psd95_coordinates, pixel_size_um, max_distance_um):
    
    """Counts the number of colocalized pre and postsynaptic spots based on the coordinates array of the local maxima detection
    
    Args:
        vlgut1_coordinates (np.array): coordinates of the local peak maxima in the vlgut1 channel
        psd95_coordinates (np.array): coordinates of the local peak maxima in the psd95 channel
        pixel_size_um (float): size of a pixel in um, based on image settings
        max_distance_um (float): value of the maximum colocalization distance between the spots of each channel in um
    
    Returns:
        colocalized spot count
    """
    
    # calculate the max distance in pixels with the pixel_size_um and max_distance_um
    max_distance_px = max_distance_um/pixel_size_um
    
    # Calculate pairwise distances between spots in vlgut1 and psd95 channel
    distances_vglut1_to_psd95 = cdist(vglut1_coordinates, psd95_coordinates)
    distances_psd95_to_vglut1 = cdist(psd95_coordinates, vglut1_coordinates)

    # Find unique colocalized spots
    colocalized_spots = set()

    # Iterate over distances from vglut1_coordinates to psd95_coordinates
    for i in range(len(vglut1_coordinates)):
        # Check if the current spot in vlgut1 has nearby spots in psd95
        colocalized_indices = [j for j, distance in enumerate(distances_vglut1_to_psd95[i, :]) if distance <= max_distance_px]
        for j in colocalized_indices:
            colocalized_spots.add((i, j))

    # Iterate over distances from psd95_coordinates to vglut1_coordinates
    for i in range(len(psd95_coordinates)):
        # Check if the current spot in Channel 2 has nearby spots in Channel 1
        colocalized_indices = [j for j, distance in enumerate(distances_psd95_to_vglut1[i, :]) if distance <= max_distance_px]
        for j in colocalized_indices:
            colocalized_spots.add((j, i))

    # Get the count of unique colocalized spots
    colocalized_spot_count = len(colocalized_spots)
    
    return colocalized_spot_count

In [ ]:
# writing a wrapper function for the two functions, which is easier to implement for the bayesian optimization
def comb_func(vglut1_preprocessed, psd95_preprocessed, vglut1_distance, psd95_distance, vglut1_threshold, psd95_threshold, pixel_size_um, max_distance_um):
    
    # probing the first function to get the parameters as input for the second function
    vglut1_coord, psd95_coord, psd95_rot_coord = local_peak_detection(vglut1_preprocessed, psd95_preprocessed, vglut1_distance, psd95_distance, vglut1_threshold, psd95_threshold)
    
    # probing the second function where psd95 is not rotated, the actual condition
    colocalized_spot_count = count_coloc_spots(vglut1_coord, psd95_coord, pixel_size_um, max_distance_um)
    # probing the second function where psd95 is rotated, the internal control condition
    colocalized_spot_count_rot = count_coloc_spots(vglut1_coord, psd95_rot_coord, pixel_size_um, max_distance_um)
    
    # getting the scale differences between the spot count of the normal situation and psd95 rotated
    scaled_difference = (colocalized_spot_count - colocalized_spot_count_rot) / max(colocalized_spot_count, 1)
    
    return colocalized_spot_count, colocalized_spot_count_rot, scaled_difference

In [ ]:
import optuna

# setting the static variables
vglut1_preprocessed = pre_1 # Assign vglut1_pre to vglut1_preprocessed
psd95_preprocessed = post_1  # Assign psd95_pre to psd95_preprocessed
pixel_size_um = pixel_size_um  # Assign pixel_size to pixel_size_um


def objective(trial):
    
    vglut1_distance = trial.suggest_int("vglut1_distance", 1, 3)
    psd95_distance = trial.suggest_int("psd95_distance", 1, 3)
    vglut1_threshold = trial.suggest_float("vlgut1_threshold", 100, 200)
    psd95_threshold = trial.suggest_float("psd95_threshold", 50, 150)
    max_distance_um = trial.suggest_float("max_distance_um", 0.01, 1)

    _, _, scaled_spot_count_dif = comb_func(
        vglut1_preprocessed = vglut1_preprocessed,
        psd95_preprocessed = psd95_preprocessed,        
        vglut1_distance =vglut1_distance, 
        psd95_distance = psd95_distance,
        vglut1_threshold = vglut1_threshold,
        psd95_threshold = psd95_threshold,
        pixel_size_um = pixel_size_um,
        max_distance_um = max_distance_um
    )

    return scaled_spot_count_dif 


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=500)
print(study.best_trial.value)

In [ ]:
from optuna.visualization import plot_contour
from optuna.visualization import plot_edf
from optuna.visualization import plot_intermediate_values
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_rank
from optuna.visualization import plot_slice
from optuna.visualization import plot_timeline



plot_rank(study)